# nbadb — Kaggle Pipeline
Automated NBA database update notebook. Runs on Kaggle infrastructure.

In [ ]:
!pip install nbadb -q

In [ ]:
import os
from pathlib import Path

# Load Kaggle secrets
os.environ["NBADB_DATA_DIR"] = "/kaggle/working/nbadb"
os.environ["NBADB_LOG_DIR"] = "/kaggle/working/logs"

from nbadb.core.config import get_settings
settings = get_settings()
print(f"Data dir: {settings.data_dir}")
print(f"Formats: {settings.formats}")

In [ ]:
from nbadb.kaggle.notebook import determine_update_mode

mode = determine_update_mode()
print(f"Update mode: {mode}")

In [ ]:
from nbadb.kaggle.client import KaggleClient

client = KaggleClient()
try:
    download_path = client.download()
    print(f"Downloaded to: {download_path}")
except Exception as e:
    print(f"No existing dataset or download failed: {e}")
    print("Will do full build")
    mode = "full"

In [ ]:
import subprocess
result = subprocess.run(
    ["nbadb", mode, "--verbose"],
    capture_output=True, text=True, timeout=14400
)
print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
if result.returncode != 0:
    print(f"STDERR: {result.stderr[-1000:]}")

In [ ]:
from nbadb.core.db import DBManager
from pathlib import Path

db = DBManager()
db.init()

# Check row counts
tables = db.duckdb.execute(
    "SELECT table_name FROM _pipeline_metadata ORDER BY table_name"
).fetchall()
print(f"Tables loaded: {len(tables)}")
for (table,) in tables[:10]:
    count = db.duckdb.execute(
        f"SELECT row_count FROM _pipeline_metadata WHERE table_name = '{table}'"
    ).fetchone()
    print(f"  {table}: {count[0]:,} rows" if count else f"  {table}: N/A")
db.close()

In [ ]:
client.ensure_metadata()
client.upload(version_notes=f"Automated {mode} update via nbadb")
print("Upload complete!")

## Summary
Pipeline complete. Check the [dataset page](https://www.kaggle.com/datasets/wyattowalsh/basketball) for updated data.